In [1]:
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.anthropic import AnthropicModel
from anthropic import Anthropic
import os

from rag_helper import RAGHelper
from ingest import load_faq_data, build_index
import json

In [2]:
load_dotenv()  # Load environment variables from .env file
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [3]:
model = AnthropicModel("claude-haiku-4-5-20251001")

In [4]:
agent = Agent(model=model, system_prompt="You are a helpful assistant.")

In [5]:
documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use ONLY the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [7]:
agent = Agent(model=model, system_prompt=instructions)

In [8]:
# Register the search function as a tool using a decorator
@agent.tool_plain
def search_tool(query: str) -> list[dict]:
    """Search the FAQ database for entries matching the query."""
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

In [9]:
result = await agent.run("How do I run Olama locally?")
print(result)

AgentRunResult(output='Based on the FAQ, here\'s how to run Ollama locally:\n\n## Installation\n\n1. **Visit the download page**: Go to [https://ollama.com/download](https://ollama.com/download) and choose your operating system:\n   - **macOS**: Download the `.pkg` and install it\n   - **Windows**: Download the `.msi` and install it\n   - **Linux**: Run this command in the terminal:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n## Running Ollama\n\n2. **Start a model**: Open a terminal and type:\n   ```bash\n   ollama run llama3\n   ```\n   This will:\n   - Download the LLaMA 3 model (~4GB)\n   - Start the model locally\n   - Open a chat-like interface where you can type questions\n\n3. **Test the server**: Run this command to verify Ollama is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should receive a JSON response with available models.\n\n## Using Ollama with Python\n\n4. **Install the Python client**:\n   ```bash\n   pip in

In [10]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content="You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse ONLY the facts from the CONTEXT when answering the QUESTION.", timestamp=datetime.datetime(2026, 9, 5, 15, 13, 52, 975585, tzinfo=datetime.timezone.utc)), UserPromptPart(content='How do I run Olama locally?', timestamp=datetime.datetime(2026, 9, 5, 15, 13, 52, 975596, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 5, 15, 13, 52, 976217, tzinfo=datetime.timezone.utc), run_id='01a07222-1b47-7296-bf6a-041e836bfe1a', conversation_id='01a07222-1b47-7296-bf6a-041f2836378d'),
 ModelResponse(parts=[TextPart(content="I'll search the FAQ database for information about running Ollama locally."), ToolCallPart(tool_name='search_tool', args={'query': 'run Ollama locally'}, tool_call_id='toolu_01D1sDcUgh3W3mK6GPeXByxY')], usage=RequestUsage(cost=Decimal('0.000981'), details={'input_tokens': 616, 'output_tokens': 73, 'cache_crea

In [11]:
# A way to keep the conversation with the LLM
result2 = await agent.run(
    "How do I run a different model?",
    message_history=result.all_messages(),
)

print(result2)

AgentRunResult(output='Based on the FAQ, the way to run a different model with Ollama is to use the same command structure but specify the model name instead of `llama3`.\n\nFrom the installation instructions, the general command is:\n\n```bash\nollama run <model-name>\n```\n\nSo for example, instead of:\n```bash\nollama run llama3\n```\n\nYou would run:\n```bash\nollama run <your-desired-model>\n```\n\nThe same applies when using the Python client — just change the `model` parameter:\n\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'<your-desired-model>\',\n    messages=[{"role": "user", "content": your_prompt}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\nHowever, the FAQ doesn\'t provide a list of available models or specific examples of other models you can run. For a complete list of available models, you would need to check the [Ollama website](https://ollama.com) directly.')
